# Contrôle du nettoyage DVF

Entrée : `data/processed/dvf_appartements.parquet`, produit par `scripts/02_clean_dvf.py` (règles dans l'en-tête du script).

In [1]:
import duckdb, geopandas as gpd, pandas as pd
g = gpd.read_parquet('../data/processed/dvf_appartements.parquet')
con = duckdb.connect()
raw = """read_csv('../data/raw/dvf/*/*.csv.gz', all_varchar=true, union_by_name=true)"""
print(g.crs.to_string(), g.shape, 'id_mutation unique :', g.id_mutation.is_unique)

EPSG:2154 (354355, 22) id_mutation unique : True


## 1. Entonnoir : combien de mutations chaque règle retire

L'effet du dédoublonnage (règle 2) est mesuré en recomptant la règle 3 sans lui.

In [2]:
con.sql(f"""create table b as select id_mutation, type_local, id_parcelle, lot1_numero, surface_reelle_bati,
  cast(valeur_fonciere as double) v, cast(surface_reelle_bati as double) s, nature_mutation
  from {raw} where nature_mutation in ('Vente', $$Vente en l'état futur d'achèvement$$)""")
con.sql("""create table m as select id_mutation,
  count(*) filter (where type_local = 'Appartement') n_app_brut,
  count(distinct (id_parcelle, lot1_numero, surface_reelle_bati)) filter (where type_local = 'Appartement') n_app,
  count(*) filter (where type_local in ('Maison', 'Local industriel. commercial ou assimilé')) n_autre
  from b group by 1""")
pd.DataFrame({
  'mutations Vente + VEFA': con.sql('select count(*) from m').fetchone(),
  '1 appart, sans maison/local (sans dédoublonnage)': con.sql('select count(*) from m where n_app_brut = 1 and n_autre = 0').fetchone(),
  '1 appart, sans maison/local (avec dédoublonnage)': con.sql('select count(*) from m where n_app = 1 and n_autre = 0').fetchone(),
  'après surface >= 9 m² et valeur > 0': [len(g)],
  'dont non signalées aberrantes': [(~g.aberrant).sum()],
  'dont non aberrantes et géolocalisées': [(~g.aberrant & g.coord_valide).sum()],
}, index=['mutations']).T

,mutations
mutations Vente + VEFA,547930
"1 appart, sans maison/local (sans dédoublonnage)",355291
"1 appart, sans maison/local (avec dédoublonnage)",355663
après surface >= 9 m² et valeur > 0,354355
dont non signalées aberrantes,346241
dont non aberrantes et géolocalisées,344257


## 2. Signalement des valeurs aberrantes

Règle : prix/m² hors de [0,3 ; 3] × la médiane de la commune pour le même type (ancien/VEFA).

In [3]:
pd.crosstab([g.code_departement, g.type_vente], g.aberrant, margins=True)

aberrant                      False  True     All
code_departement type_vente                      
75               ancien      143045  5618  148663
                 vefa           259     0     259
92               ancien       80170  1308   81478
                 vefa          9414    13    9427
93               ancien       45012   596   45608
                 vefa          8707    13    8720
94               ancien       53494   563   54057
                 vefa          6140     3    6143
All                          346241  8114  354355

In [4]:
cols = ['date_mutation', 'type_vente', 'nom_commune', 'valeur_fonciere', 'surface_reelle_bati', 'n_dependances', 'prix_m2', 'ratio_median_ref']
a = g[g.aberrant].sort_values('ratio_median_ref')
pd.concat([a.head(5), a[a.ratio_median_ref.between(0.2, 0.3)].head(5), a[a.ratio_median_ref.between(3, 4)].head(5), a.tail(5)])[cols]

,date_mutation,type_vente,nom_commune,valeur_fonciere,surface_reelle_bati,n_dependances,prix_m2,ratio_median_ref
162513,2022-12-14,ancien,Paris 11e Arrondissement,1.000000e+00,460.0,0,2.173913e-03,2.091928e-07
351755,2025-12-17,ancien,Paris 4e Arrondissement,1.000000e+00,230.0,0,4.347826e-03,3.371783e-07
318025,2025-06-19,ancien,Paris 9e Arrondissement,1.000000e+00,191.0,0,5.235602e-03,4.687110e-07
233437,2024-01-15,ancien,Paris 9e Arrondissement,1.000000e+00,189.0,0,5.291005e-03,4.736709e-07
161505,2022-12-09,ancien,Paris 16e Arrondissement,1.000000e+00,178.0,0,5.617978e-03,4.988938e-07
244186,2024-03-28,ancien,Noisy-le-Grand,6.875000e+04,80.0,0,8.593750e+02,2.002128e-01
213736,2023-09-13,ancien,Paris 17e Arrondissement,3.000000e+04,14.0,0,2.142857e+03,2.004608e-01
297467,2025-02-26,ancien,Paris 20e Arrondissement,2.479951e+05,140.0,0,1.771394e+03,2.005351e-01
123146,2022-06-28,ancien,Paris 4e Arrondissement,1.655000e+05,64.0,1,2.585938e+03,2.005421e-01
331609,2025-09-01,ancien,Saint-Ouen-sur-Seine,3.600000e+04,27.0,1,1.333333e+03,2.006689e-01


## 3. Distribution après filtrage

In [5]:
ok = g[~g.aberrant]
ok.groupby(['code_departement', 'type_vente']).prix_m2.describe(percentiles=[0.01, 0.5, 0.99]).round(0)

count     mean     std     min      1%  \
code_departement type_vente                                              
75               ancien      143045.0  10728.0  3257.0  2608.0  4500.0   
                 vefa           259.0  11554.0  3410.0  4545.0  6510.0   
92               ancien       80170.0   7077.0  2342.0  1186.0  2713.0   
                 vefa          9414.0   7338.0  2048.0  1697.0  3885.0   
93               ancien       45012.0   4381.0  1859.0   577.0  1360.0   
                 vefa          8707.0   4997.0  1315.0  1381.0  2855.0   
94               ancien       53494.0   5288.0  2136.0   944.0  1998.0   
                 vefa          6140.0   5832.0  1739.0  2098.0  3247.0   

                                 50%      99%      max  
code_departement type_vente                             
75               ancien      10345.0  22311.0  43125.0  
                 vefa        11305.0  23991.0  27500.0  
92               ancien       6818.0  14077.0  31128.0  
                 vefa         7099.0  13535.0  33615.0  
93               ancien       4004.0   9200.0  20000.0  
                 vefa         4725.0   8436.0  17436.0  
94               ancien       4925.0  11424.0  26545.0  
                 vefa         5500.0  11576.0  17539.0

In [6]:
ok.pivot_table(index='annee', columns=['code_departement', 'type_vente'], values='prix_m2', aggfunc='median').round(0)

code_departement       75               92              93              94  \
type_vente         ancien     vefa  ancien    vefa  ancien    vefa  ancien   
annee                                                                        
2021              10908.0  11314.0  7097.0  6971.0  3967.0  4625.0  4958.0   
2022              10770.0  11986.0  7159.0  7242.0  4176.0  4757.0  5195.0   
2023              10225.0  11667.0  6829.0  7050.0  4056.0  4714.0  4962.0   
2024               9649.0  10218.0  6400.0  7148.0  3857.0  4781.0  4672.0   
2025               9762.0   8670.0  6441.0  7139.0  4000.0  5157.0  4750.0   

code_departement          
type_vente          vefa  
annee                     
2021              5633.0  
2022              5653.0  
2023              5476.0  
2024              5230.0  
2025              5049.0

In [7]:
ok.pivot_table(index='annee', columns='code_departement', values='id_mutation', aggfunc='count')

code_departement,75,92,93,94
annee,,,,
2021,30490,21773,14102,14351
2022,33185,21431,13397,14402
2023,26882,16140,9607,11039
2024,24419,14015,7819,9254
2025,28328,16225,8794,10588


## 4. Géolocalisation

In [8]:
print('sans point :', (~g.coord_valide).sum(), '/', len(g))
print('emprise (WGS84) :', g[g.coord_valide].to_crs(4326).total_bounds.round(3))

sans point : 2014 / 354355


emprise (WGS84) : [ 2.147 48.694  2.595 48.982]
